# Recolección de datos y Análisis Exploratorio (EDA)

Este notebook documenta el proceso de recolección y preparación de los datos utilizados en el caso de estudio de Lima Metropolitana. Se describen las fuentes, la estructura de cada archivo, la limpieza inicial y la exploración analítica.

## Objetivo
- Revisar las fuentes de datos disponibles.
- Explicar el proceso de recolección.
- Validar la estructura de los archivos.
- Realizar un EDA inicial para identificar patrones, calidad y relaciones clave.


In [ ]:
from pathlib import Path
import sys

repo_base = 'https://raw.githubusercontent.com/GianfrancoManrique/intro_data_science_1/main/'
project_root = Path.cwd()

if 'google.colab' in sys.modules:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception:
        pass

    base_drive = Path('/content/drive')
    candidates = [
        base_drive / 'MyDrive' / 'intro_data_science_1',
        base_drive / 'intro_data_science_1',
        base_drive / 'MyDrive',
        Path('/content/intro_data_science_1'),
        Path('/content'),
    ]
    project_root = next(
        (p for p in candidates if (p / 'data').exists() or (p / 'results').exists()),
        Path.cwd(),
    )
else:
    if not (project_root / 'data').exists():
        parent = project_root.parent
        if (parent / 'data').exists():
            project_root = parent

data_raw = project_root / 'data' / 'raw'
results_dir = project_root / 'results' / 'eda'
results_dir.mkdir(parents=True, exist_ok=True)

local_csv = data_raw / 'DATASET_Denuncias_Policiales_Enero 2018_Julio2026_LIMA.csv'
local_xlsx = data_raw / 'POBLACION_NBI_LIMA.xlsx'
csv_url = repo_base + 'data/raw/DATASET_Denuncias_Policiales_Enero%202018_Julio2026_LIMA.csv'
xlsx_url = repo_base + 'data/raw/POBLACION_NBI_LIMA.xlsx'

print('Proyecto detectado en:', project_root)
print('Archivo CSV local presente:', local_csv.exists())
print('Archivo XLSX local presente:', local_xlsx.exists())
print('Ruta de resultados:', results_dir)
print('URL del CSV remoto:', csv_url)
print('URL del XLSX remoto:', xlsx_url)

## 1. Fuentes de datos

El conjunto de trabajo está compuesto por dos fuentes principales:

- `DATASET_Denuncias_Policiales_Enero 2018_Julio2026_LIMA.csv`: archivo principal con denuncias policiales en Lima.
- `POBLACION_NBI_LIMA.xlsx`: datos demográficos por distrito/ubigeo, incluyendo población total y porcentaje de población en condiciones de NBI.

Estas fuentes se recopilaron y se integraron por `UBIGEO` y `DISTRITO` para construir una base analítica más completa.

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

repo_base = 'https://raw.githubusercontent.com/GianfrancoManrique/intro_data_science_1/main/'
csv_url = repo_base + 'data/raw/DATASET_Denuncias_Policiales_Enero%202018_Julio2026_LIMA.csv'
xlsx_url = repo_base + 'data/raw/POBLACION_NBI_LIMA.xlsx'

project_root = Path.cwd()
if not (project_root / 'data').exists():
    candidate = project_root.parent
    if (candidate / 'data').exists():
        project_root = candidate

data_raw = project_root / 'data' / 'raw'
results_dir = project_root / 'results' / 'eda'
results_dir.mkdir(parents=True, exist_ok=True)

local_csv = data_raw / 'DATASET_Denuncias_Policiales_Enero 2018_Julio2026_LIMA.csv'
local_xlsx = data_raw / 'POBLACION_NBI_LIMA.xlsx'


def load_csv_or_remote(path_local, url):
    return pd.read_csv(path_local if path_local.exists() else url)


def load_excel_or_remote(path_local, url):
    return pd.read_excel(path_local if path_local.exists() else url)


df_denuncias = load_csv_or_remote(local_csv, csv_url)
df_demo = load_excel_or_remote(local_xlsx, xlsx_url)

print('CSV - filas, columnas:', df_denuncias.shape)
print('XLSX - filas, columnas:', df_demo.shape)
print('\nVista preliminar del CSV:')
display(df_denuncias.head(3))
print('\nVista preliminar del XLSX:')
display(df_demo.head(3))
print('\nColumnas del CSV:')
print(list(df_denuncias.columns))
print('\nColumnas del XLSX:')
print(list(df_demo.columns))

## 2. Proceso de recolección

La recolección se realizó a partir de dos fuentes de datos institucionales y territoriales:

1. Datos de denuncias policiales, con información temporal y geográfica por distrito/ubigeo.
2. Datos demográficos y socioeconómicos por ubigeo para contextualizar el fenómeno.

Durante el proceso se verificó:
- la existencia de archivos principales (`.csv`, `.xlsx`),
- la estructura esperada de las columnas,
- la necesidad de homogeneizar códigos geográficos (`UBIGEO`) para unir ambos datasets correctamente,
- la necesidad de filtrar años parciales o irregulares para análisis comparables.

In [ ]:
# Validación inicial de calidad
print('Duplicados en denuncias:', df_denuncias.duplicated().sum())
print('Duplicados en población:', df_demo.duplicated().sum())
print('\nNulos por columna en denuncias:')
print(df_denuncias.isna().sum().head(15))
print('\nNulos por columna en población:')
print(df_demo.isna().sum().head(15))

## 3. Preparación de datos

Antes de combinar la información, se normaliza la clave geográfica para evitar inconsistencias en los códigos y garantizar un merge consistente por ubigeo.

In [ ]:
# Normalización de UBIGEO
df_denuncias['UBIGEO_HECHO'] = df_denuncias['UBIGEO_HECHO'].astype(str).str.zfill(6)
if 'UBIGEO' in df_demo.columns:
    df_demo['UBIGEO'] = df_demo['UBIGEO'].astype(str).str.zfill(6)

print('Ejemplo de UBIGEO normalizado:')
print(df_denuncias['UBIGEO_HECHO'].head().tolist())
print('\nCantidad de UBIGEOs únicos en denuncias:', df_denuncias['UBIGEO_HECHO'].nunique())
print('Cantidad de UBIGEOs únicos en población:', df_demo['UBIGEO'].nunique())

## 4. Exploración inicial (EDA)

Se realiza una revisión descriptiva para comprender distribución, tendencias temporales y relación con variables socioeconómicas.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns

# Distribución anual de denuncias
if 'ANIO' in df_denuncias.columns:
    anual = df_denuncias.groupby('ANIO').size().sort_index()
    plt.figure(figsize=(12, 5))
    anual.plot(kind='bar', color='steelblue')
    plt.title('Cantidad de denuncias por año')
    plt.xlabel('Año')
    plt.ylabel('Número de denuncias')
    plt.tight_layout()
    plt.show()

# Distribución por modalidad (si existe)
if 'P_MODALIDADES' in df_denuncias.columns:
    modal = df_denuncias['P_MODALIDADES'].value_counts().head(10)
    plt.figure(figsize=(10, 5))
    sns.barplot(x=modal.values, y=modal.index, palette='viridis')
    plt.title('Top modalidades de delito')
    plt.xlabel('Cantidad')
    plt.ylabel('Modalidad')
    plt.tight_layout()
    plt.show()

In [ ]:
%matplotlib inline

# Merge demostrativo para observar contexto geográfico y socioeconómico
merged = df_denuncias.merge(df_demo, left_on='UBIGEO_HECHO', right_on='UBIGEO', how='left')

cols = [c for c in ['UBIGEO_HECHO', 'DISTRITO', 'cantidad', 'POBLACION_TOTAL', 'PORC_NBI'] if c in merged.columns]
print('Muestra del merge:')
display(merged[cols].head(10))

numeric = merged.select_dtypes(include='number').columns.tolist()
numeric = [c for c in numeric if c in ['cantidad', 'POBLACION_TOTAL', 'PORC_NBI']]
if len(numeric) >= 2:
    corr = merged[numeric].corr()
    plt.figure(figsize=(7, 5))
    sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1, center=0)
    plt.title('Correlación entre variables clave')
    plt.tight_layout()
    plt.show()

## 5. Visualización de patrones (EDA gráfico)

En esta sección se generan gráficos para visualizar la evolución temporal, la distribución de los registros y la relación entre denuncias y condiciones socioeconómicas.

## Exploración univariada

Se revisa la distribución individual de las variables clave para identificar asimetrías, presencia de valores extremos y patrones generales dentro de la base analítica.

In [ ]:
# Exploración univariada
import pandas as pd
from pathlib import Path

project_root = Path.cwd()
if not (project_root / 'data').exists():
    candidate = project_root.parent
    if (candidate / 'data').exists():
        project_root = candidate

univ_path = project_root / 'data' / 'processed' / 'datos_temporales.csv'
repo_base = 'https://raw.githubusercontent.com/GianfrancoManrique/intro_data_science_1/main/'
univ_url = repo_base + 'data/processed/datos_temporales.csv'

if univ_path.exists():
    df_univ = pd.read_csv(univ_path)
else:
    df_univ = pd.read_csv(univ_url)

numeric_cols = ['POBLACION_TOTAL', 'PORC_NBI', 'TOTAL_DELITOS', 'TASA_DELITOS_10K']
missing = [col for col in numeric_cols if col not in df_univ.columns]
if not missing:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    for ax, col in zip(axes.flat, numeric_cols):
        sns.histplot(df_univ[col].dropna(), bins=25, kde=True, ax=ax, color='#4C78A8', edgecolor='black')
        ax.set_title(f'Distribución de {col}')
        ax.set_xlabel(col)
        ax.set_ylabel('Frecuencia')
    plt.tight_layout()
    plt.show()
else:
    print('Faltan columnas para la exploración univariada:', missing)

In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

repo_base = 'https://raw.githubusercontent.com/GianfrancoManrique/intro_data_science_1/main/'
project_root = Path.cwd()

if not (project_root / 'data').exists():
    candidate = project_root.parent
    if (candidate / 'data').exists():
        project_root = candidate

processed_path = project_root / 'data' / 'processed' / 'datos_limpios_preparados.csv'
clean_path = project_root / 'data' / 'clean' / 'datos_merged.csv'
results_dir = project_root / 'results' / 'eda'
results_dir.mkdir(parents=True, exist_ok=True)

processed_url = repo_base + 'data/processed/datos_limpios_preparados.csv'
clean_url = repo_base + 'data/clean/datos_merged.csv'

if processed_path.exists():
    df_eda = pd.read_csv(processed_path)
    print('Dataset EDA cargado desde local:', df_eda.shape)
elif clean_path.exists():
    df_eda = pd.read_csv(clean_path)
    print('Dataset EDA cargado desde clean local:', df_eda.shape)
else:
    df_eda = pd.read_csv(processed_url)
    print('Dataset EDA cargado desde GitHub:', df_eda.shape)

print('Columnas disponibles:', list(df_eda.columns[:10]))
print('Listo para generar cada gráfico por separado.')

In [ ]:
# 1. Top 10 distritos con más denuncias
if {'DISTRITO', 'TOTAL_DELITOS'}.issubset(df_eda.columns):
    top_distritos = df_eda[['DISTRITO', 'TOTAL_DELITOS']].sort_values('TOTAL_DELITOS', ascending=False).head(10)
    plt.figure(figsize=(10, 6))
    sns.barplot(data=top_distritos, x='TOTAL_DELITOS', y='DISTRITO', palette='magma')
    plt.title('Top 10 distritos con más denuncias')
    plt.xlabel('Total de denuncias')
    plt.ylabel('Distrito')
    plt.tight_layout()
    plt.show()

In [ ]:
# 2. Relación entre % NBI y denuncias por distrito
if {'DISTRITO', 'TOTAL_DELITOS', 'PORC_NBI_2017'}.issubset(df_eda.columns):
    distrito_agg = df_eda[['DISTRITO', 'TOTAL_DELITOS', 'PORC_NBI_2017']].copy()
    plt.figure(figsize=(9, 6))
    sns.scatterplot(data=distrito_agg, x='PORC_NBI_2017', y='TOTAL_DELITOS', size='TOTAL_DELITOS', hue='TOTAL_DELITOS', palette='viridis', sizes=(60, 300), alpha=0.8)
    plt.title('% NBI vs denuncias por distrito')
    plt.xlabel('% NBI')
    plt.ylabel('Total de denuncias')
    plt.tight_layout()
    plt.show()

In [ ]:
# 3. Matriz de correlación entre variables numéricas clave
numeric_cols = [c for c in ['TOTAL_DELITOS', 'POBLACION_TOTAL', 'PORC_NBI_2017'] if c in df_eda.columns]
if len(numeric_cols) >= 2:
    corr = df_eda[numeric_cols].corr()
    plt.figure(figsize=(7, 5))
    sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1, center=0)
    plt.title('Correlación entre variables clave')
    plt.tight_layout()
    plt.show()

## Análisis temporal

La evolución temporal permite verificar la tendencia anual de la incidencia delictiva y compararla con cambios en la población y la vulnerabilidad social.

In [ ]:
# Análisis temporal
import pandas as pd
from pathlib import Path

project_root = Path.cwd()
if not (project_root / 'data').exists():
    candidate = project_root.parent
    if (candidate / 'data').exists():
        project_root = candidate

temporal_path = project_root / 'data' / 'processed' / 'datos_temporales.csv'
repo_base = 'https://raw.githubusercontent.com/GianfrancoManrique/intro_data_science_1/main/'
temporal_url = repo_base + 'data/processed/datos_temporales.csv'

if temporal_path.exists():
    df_temporal = pd.read_csv(temporal_path)
else:
    df_temporal = pd.read_csv(temporal_url)

required_cols = {'ANIO', 'TOTAL_DELITOS', 'TASA_DELITOS_10K', 'POBLACION_TOTAL', 'PORC_NBI'}
if required_cols.issubset(df_temporal.columns):
    temporal = (
        df_temporal[df_temporal['ANIO'] < 2026]
        .groupby('ANIO', as_index=False)
        .agg(
            TOTAL_DELITOS=('TOTAL_DELITOS', 'sum'),
            TASA_PROMEDIO_10K=('TASA_DELITOS_10K', 'mean'),
            POBLACION_TOTAL=('POBLACION_TOTAL', 'sum'),
            PORC_NBI_PROMEDIO=('PORC_NBI', 'mean')
        )
        .sort_values('ANIO')
    )
    if not temporal.empty:
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        sns.lineplot(data=temporal, x='ANIO', y='TOTAL_DELITOS', marker='o', ax=axes[0, 0], color='#4C78A8')
        axes[0, 0].set_title('Total de delitos por año')
        axes[0, 0].set_xlabel('Año')
        axes[0, 0].set_ylabel('Total de delitos')

        sns.lineplot(data=temporal, x='ANIO', y='TASA_PROMEDIO_10K', marker='o', ax=axes[0, 1], color='#4C78A8')
        axes[0, 1].set_title('Tasa promedio de delitos por 10k por año')
        axes[0, 1].set_xlabel('Año')
        axes[0, 1].set_ylabel('Tasa promedio')

        sns.lineplot(data=temporal, x='ANIO', y='POBLACION_TOTAL', marker='o', ax=axes[1, 0], color='#4C78A8')
        axes[1, 0].set_title('Población total por año')
        axes[1, 0].set_xlabel('Año')
        axes[1, 0].set_ylabel('Población total')

        sns.lineplot(data=temporal, x='ANIO', y='PORC_NBI_PROMEDIO', marker='o', ax=axes[1, 1], color='#4C78A8')
        axes[1, 1].set_title('% NBI promedio por año')
        axes[1, 1].set_xlabel('Año')
        axes[1, 1].set_ylabel('% NBI')

        plt.tight_layout()
        plt.show()
    else:
        print('No hay datos temporales para graficar.')
else:
    print('Faltan columnas requeridas para el análisis temporal:', sorted(required_cols - set(df_temporal.columns)))

## 6. Resultados preliminares

A partir de la revisión inicial se observa que:

- la base de denuncias contiene una estructura temporal y geográfica adecuada para análisis por año y distrito;
- la información demográfica permite contextualizar la incidencia por ubigeo;
- hay necesidad de validar la consistencia de los códigos de ubigeo antes del análisis final;
- la comparación entre denuncias y niveles de NBI puede ayudar a detectar relaciones de riesgo territorial.

## Resultados descriptivos clave

Los indicadores descriptivos muestran que los distritos presentan diferencias relevantes en población, porcentaje de NBI, total de delitos y tasa ajustada por 10,000 habitantes.

## 7. Conclusión

El conjunto de datos recolectado resulta adecuado para un análisis exploratorio inicial. La combinación de información policial y demográfica permite construir un relato más sólido sobre la distribución territorial de los delitos y sus relaciones con condiciones socioeconómicas.

El siguiente paso recomendado es la limpieza definitiva, la agregación por distrito-año y la preparación de indicadores estandarizados para análisis más profundos.

### Inspección multivariada

La revisión conjunta de variables permite explorar asociaciones entre población, vulnerabilidad social y criminalidad, reconociendo la naturaleza descriptiva del análisis.

In [ ]:
# Inspección multivariada
import pandas as pd
from pathlib import Path

project_root = Path.cwd()
if not (project_root / 'data').exists():
    candidate = project_root.parent
    if (candidate / 'data').exists():
        project_root = candidate

temporal_path = project_root / 'data' / 'processed' / 'datos_temporales.csv'
repo_base = 'https://raw.githubusercontent.com/GianfrancoManrique/intro_data_science_1/main/'
temporal_url = repo_base + 'data/processed/datos_temporales.csv'

if temporal_path.exists():
    df_multivar = pd.read_csv(temporal_path)
else:
    df_multivar = pd.read_csv(temporal_url)

required_cols = ['POBLACION_TOTAL', 'PORC_NBI', 'TOTAL_DELITOS', 'TASA_DELITOS_10K']
missing = [col for col in required_cols if col not in df_multivar.columns]
if not missing:
    pair_df = df_multivar[required_cols].dropna().copy()
    if not pair_df.empty:
        pairplot = sns.pairplot(pair_df, diag_kind='kde', height=2.2, plot_kws={'alpha': 0.8})
        pairplot.fig.suptitle('Pairplot de variables clave', y=1.02)
        plt.show()
    else:
        print('No hay datos suficientes para la inspección multivariada.')
else:
    print('Faltan columnas para la inspección multivariada:', missing)